# ⚡ VRP Telemetry Data Cleaning & Preprocessing Pipeline
### Production-Grade ML Preprocessing for JEV Model Analysis

This walkthrough notebook demonstrates each modular step of the VRP telemetry data engineering pipeline:
1. **Data Ingestion & Timestamp Standardization**: Aligning mismatched CSV columns and parsing mixed UTC timestamps.
2. **Data Cleaning & EV Domain Guards**: Enforcing physical constraints (speed limits, non-negative cell voltage delta, SOC bounds).
3. **Outlier Detection & Capping**: IQR-based winsorization to preserve time-series continuity.
4. **Exploratory Data Analysis (EDA)**: Missingness distributions and correlation matrices.
5. **Leak-Free Data Splitting**: Chronological 70% / 15% / 15% split prior to fitting transformers.
6. **Feature Selection**: Zero-variance removal and collinearity (|r| > 0.90) elimination strictly on the training split.
7. **Scikit-Learn Preprocessing Pipeline**: ColumnTransformer with median imputation, scaling, and categorical encoding.
8. **Export & Preprocessor Serialization**: Saving `train.csv`, `val.csv`, `test.csv`, and `preprocessor.joblib`.

In [1]:
import os
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    DEFAULT_RAW_FILE,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    PLOTS_DIR,
    MISSING_COL_THRESHOLD,
    CORR_THRESHOLD,
    IQR_MULTIPLIER,
    DOMAIN_GUARDS
)
from src.data_loader import load_dataset
from src.cleaning import clean_dataset
from src.analysis import run_full_analysis
from src.pipeline import split_data
from src.feature_selection import FeatureSelector
from src.preprocessing import VRPPreprocessor

print(f"Project root: {PROJECT_ROOT}")

Project root: E:\vrp-data-pipeline


## 1. Data Ingestion & Timestamp Standardization
We ingest the raw EV telemetry CSV (`2026-10-27_EV_Analysis_Data.csv`).
Mixed ISO timestamps (both `+00:00` and `Z`) and epoch timestamps are converted into a canonical `ts_utc` column and sorted chronologically.

In [2]:
input_csv = DEFAULT_RAW_FILE
raw_df, loader_meta = load_dataset(input_csv)

print(f"Loaded {loader_meta['initial_rows']} rows and {loader_meta['initial_cols']} columns.")
print(f"Timestamp detected: {loader_meta['has_timestamp']}, range: {loader_meta['time_range']}")
raw_df[["source_file", "ts_utc", "soc", "pack_voltage", "power_kw", "speed_valid"]].head()

Loaded 5125 rows and 49 columns.
Timestamp detected: True, range: {'start': '2026-10-27 04:39:24.159000+00:00', 'end': '2026-10-27 18:03:38.647000+00:00'}


## 2. Data Cleaning & EV Physical Domain Guards
In this stage:
- Column names are converted to `snake_case`.
- Exact duplicates are dropped.
- Columns with missingness exceeding 60% are dropped.
- EV Physical Domain Guards are enforced:
  - `speed` clamped to [0, 150] km/h
  - `cell_delta_mv` clamped to non-negative values (minimum 0.0 mV)
  - `soc` clamped to [0, 100]%
- Continuous features are capped using the IQR rule (1.5x IQR) to mitigate extreme sensor glitches without disrupting telemetry sequence.

In [3]:
target_col = "power_kw"

cleaned_df, cleaning_audit = clean_dataset(
    df=raw_df,
    target_col=target_col,
    missing_col_threshold=MISSING_COL_THRESHOLD,
    drop_unconfirmed=False,
    iqr_multiplier=IQR_MULTIPLIER,
    outlier_strategy="cap"
)

print(f"Rows before/after cleaning: {cleaning_audit['initial_rows']} -> {cleaning_audit['final_rows']}")
print(f"Cols before/after cleaning: {cleaning_audit['initial_cols']} -> {cleaning_audit['final_cols']}")
print(f"Dropped high-missing columns (>60%): {list(cleaning_audit['dropped_high_missing_cols'].keys())}")
print(f"Domain violations corrected: {cleaning_audit['domain_violations_corrected']}")

Rows before/after cleaning: 5125 -> 5125
Cols before/after cleaning: 49 -> 42
Dropped high-missing columns (>60%): ['odometer_km', 'inverter_temp', 'motor_winding_temp', 'ccs_request_a', 'ccs_station_a', 'ccs_station_v', 'vcu_charge_limit_a']
Domain violations corrected: {'speed_raw': 19, 'cell_delta_mv': 4}


## 3. Exploratory Data Analysis & Visualizations
We generate correlation heatmaps, missingness profiles, outlier distributions, and target correlation plots.

In [4]:
analysis_results = run_full_analysis(
    df=cleaned_df,
    target_col=target_col,
    plots_dir=PLOTS_DIR,
    missing_threshold=MISSING_COL_THRESHOLD
)

print("Top 5 Features correlated with target:")
for item in analysis_results["top_features_correlated_with_target"][:5]:
    print(f"  {item['feature']:<20}: r = {item['pearson_r']:.4f}")

print(f"Plots saved in: {PLOTS_DIR}")

Top 5 Features correlated with target:
  power_calc_kw       : r = 0.9757
  pack_current        : r = 0.9753
  e_kwh               : r = 0.6845
  min_cell_id         : r = 0.4653
  motor_rpm           : r = 0.4170
Plots saved in: E:\vrp-data-pipeline\outputs\plots


## 4. Leak-Free Data Splitting (Chronological)
To prevent data leakage, data is split into **70% Train**, **15% Validation**, and **15% Test** *before* fitting any feature selectors, imputers, or scalers.

In [5]:
X_train, X_val, X_test, y_train, y_val, y_test, split_meta = split_data(
    df=cleaned_df,
    target_col=target_col,
    split_mode="chronological",
    ratios=(0.70, 0.15, 0.15)
)

print(f"Train samples: {len(X_train)} ({split_meta['train_ratio']*100:.0f}%)")
print(f"Val samples:   {len(X_val)} ({split_meta['val_ratio']*100:.0f}%)")
print(f"Test samples:  {len(X_test)} ({split_meta['test_ratio']*100:.0f}%)")

Train samples: 3587 (70%)
Val samples:   769 (15%)
Test samples:  769 (15%)


## 5. Feature Selection: Zero-Variance & Collinearity Elimination
We eliminate constant features and highly collinear pairs (|r| > 0.90) by fitting `FeatureSelector` strictly on `X_train`.

In [6]:
selector = FeatureSelector(corr_threshold=CORR_THRESHOLD)
selector.fit(X_train, y_train)

X_train_sel = selector.transform(X_train)
X_val_sel = selector.transform(X_val)
X_test_sel = selector.transform(X_test)

print(f"Zero-variance dropped: {selector.zero_variance_cols_}")
print(f"Collinear dropped (|r| > {CORR_THRESHOLD}): {selector.collinear_dropped_}")
print(f"Features retained: {len(selector.selected_features_)}")

Zero-variance dropped: []
Collinear dropped (|r| > 0.9): ['batt_temp', 'cell_delta', 'coolant_in', 'elapsed_s', 'max_cell_mv', 'min_cell_mv', 'pack_current', 'soc', 'speed_raw', 'speed_valid']
Features retained: 26


## 6. Preprocessing: Imputation & Feature Scaling
We fit `VRPPreprocessor` (SimpleImputer + StandardScaler / OneHotEncoder) strictly on `X_train_sel` and apply `.transform()` to `X_val_sel` and `X_test_sel`.

In [7]:
preprocessor = VRPPreprocessor(scaler_type="standard")
preprocessor.fit(X_train_sel, y_train)

X_train_proc = preprocessor.transform(X_train_sel)
X_val_proc = preprocessor.transform(X_val_sel)
X_test_proc = preprocessor.transform(X_test_sel)

print(f"Post-processing NaNs -> Train: {X_train_proc.isna().sum().sum()}, Val: {X_val_proc.isna().sum().sum()}, Test: {X_test_proc.isna().sum().sum()}")
print(f"Final feature shape: {X_train_proc.shape}")
X_train_proc.head()

Post-processing NaNs -> Train: 0, Val: 0, Test: 0
Final feature shape: (3587, 37)


## 7. Saving Processed Splits and Serializing Pipeline Artifact
The processed splits are saved to `data/processed/` along with the fitted `preprocessor.joblib` artifact ready for JEV model training.

In [8]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_df = X_train_proc.copy()
train_df[target_col] = y_train.values

val_df = X_val_proc.copy()
val_df[target_col] = y_val.values

test_df = X_test_proc.copy()
test_df[target_col] = y_test.values

train_df.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
val_df.to_csv(PROCESSED_DATA_DIR / "val.csv", index=False)
test_df.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)
preprocessor.save(PROCESSED_DATA_DIR / "preprocessor.joblib")

print("Pipeline walkthrough completed successfully!")
print(f"Processed splits saved in: {PROCESSED_DATA_DIR}")

Pipeline walkthrough completed successfully!
Processed splits saved in: E:\vrp-data-pipeline\data\processed
